# Phase 2: Autoencoder Training (Optimized)

---

## What This Notebook Does

In Phase 1, we converted raw `.wav` audio files into 2D **Log-Mel Spectrograms** and scaled
them to `[0, 1]`. Now we build, train, and save the **Unsupervised Autoencoder** — the brain
of our anomaly detection system.

### Key Concept: Training the Autoencoder IS Training the Anomaly Detector

There is **no separate detection model**. The autoencoder learns to reconstruct normal sounds.
At inference time, the **reconstruction error (MSE)** between input and output IS the anomaly
score:

- **Low MSE** → model reconstructed it well → sound is **normal**
- **High MSE** → model failed to reconstruct → sound is **anomalous**

### Improvements Over the Original Architecture

| Change | Old | New | Why |
|--------|-----|-----|-----|
| **Layer widths** | 1024 → 256 → 64 | 512 → 128 → 32 | Reduces parameters from 82M to ~21M — far more appropriate for 850 training samples |
| **BatchNorm** | None | After every Linear layer | Stabilizes gradients and accelerates convergence across the very wide input layer |
| **Dropout** | First encoder layer only | All hidden layers | Better Domain Shift regularization — prevents memorization at every depth |
| **Input dimension** | Hardcoded 128×313 | Dynamic from data | Automatically handles any spectrogram width (313 for 10s clips, 376 for 12s clips) |
| **Bottleneck** | 64 | 32 | Tighter compression (1,252:1) forces even more aggressive feature selection |
| **LR Scheduler** | None | ReduceLROnPlateau | Automatically reduces learning rate when validation loss plateaus |

---

## 1. Library Imports

| Library | Purpose |
|---------|--------|
| **`torch`** | Deep learning framework — tensors, autograd, GPU acceleration |
| **`torch.nn`** | Neural network layers (`Linear`, `BatchNorm1d`, `ReLU`, `Sigmoid`, `Dropout`) |
| **`torch.utils.data`** | `TensorDataset` and `DataLoader` for efficient batched training |
| **`numpy`** | Loading `.npy` spectrograms |
| **`copy`** | `deepcopy` to snapshot the best model weights during training |
| **`matplotlib`** | Plotting training/validation loss curves |

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import os
import copy
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("Libraries imported successfully.")
print(f"  PyTorch version : {torch.__version__}")
print(f"  NumPy version   : {np.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"  GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---

## 2. Hyperparameters

All tunable values are centralized here for easy experimentation.

| Parameter | Value | Reasoning |
|-----------|-------|----------|
| **BATCH_SIZE** | 64 | Large enough for stable gradient estimates, small enough to fit in GPU memory |
| **LEARNING_RATE** | 1e-3 | Standard starting point for Adam optimizer |
| **WEIGHT_DECAY** | 1e-5 | L2 regularization — penalizes large weights to prevent overfitting |
| **NUM_EPOCHS** | 100 | Maximum training duration — Early Stopping will likely trigger before this |
| **PATIENCE** | 15 | Stop if validation loss doesn't improve for 15 consecutive epochs |
| **BOTTLENECK_DIM** | 32 | Compressed representation size — 1,252:1 compression ratio |

In [ ]:
BATCH_SIZE     = 64
LEARNING_RATE  = 1e-3
WEIGHT_DECAY   = 1e-5
NUM_EPOCHS     = 100
PATIENCE       = 15
BOTTLENECK_DIM = 32

PROCESSED_DIR = os.path.join("..", "data", "processed")
MODELS_DIR    = os.path.join("..", "models")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Hyperparameters set:")
print(f"  Batch Size     : {BATCH_SIZE}")
print(f"  Learning Rate  : {LEARNING_RATE}")
print(f"  Weight Decay   : {WEIGHT_DECAY}")
print(f"  Max Epochs     : {NUM_EPOCHS}")
print(f"  Early Stop     : {PATIENCE} epochs patience")
print(f"  Bottleneck     : {BOTTLENECK_DIM} dimensions")
print(f"  Device         : {device}")

---

## 3. Improved Autoencoder Architecture

### What Changed and Why

**BatchNorm1d** normalizes each mini-batch to have zero mean and unit variance. This:
- **Stabilizes training** — prevents internal covariate shift
- **Allows higher learning rates** — gradients are better behaved
- **Acts as mild regularization** — reduces overfitting slightly

**Dropout on all hidden layers** ensures Domain Shift regularization is applied throughout
the network. Each representation level learns redundant, robust features.

**Dynamic `input_dim`** lets the same class handle `(128, 313)` and `(128, 376)` spectrograms.

```
Encoder:  input_dim → 512 → 128 → 32 (bottleneck)
Decoder:  32 → 128 → 512 → input_dim
```

In [ ]:
class AcousticAutoencoder(nn.Module):
    """
    Optimized Dense Autoencoder for Acoustic Anomaly Detection.
    """

    def __init__(self, input_dim, bottleneck_dim=32):
        super().__init__()
        self.input_dim = input_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, bottleneck_dim),
            nn.ReLU(),
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),

            nn.Linear(512, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        original_shape = x.shape
        x = x.view(original_shape[0], -1)
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded.view(original_shape)


print("AcousticAutoencoder class defined successfully.")

---

## 4. Shape Verification (Smoke Test)

Verify the architecture works for **both** spectrogram sizes:
- `(128, 313)` — bearing, fan, gearbox, slider, valve
- `(128, 376)` — ToyCar, ToyTrain

In [ ]:
for width in [313, 376]:
    input_dim = 128 * width
    model_test = AcousticAutoencoder(input_dim, BOTTLENECK_DIM).to(device)
    dummy = torch.randn(4, 128, width).to(device)
    with torch.no_grad():
        out = model_test(dummy)
    total_params = sum(p.numel() for p in model_test.parameters())
    assert dummy.shape == out.shape
    assert out.min() >= 0.0 and out.max() <= 1.0
    print(f"  Width {width}: input={dummy.shape} -> output={out.shape}  params={total_params:,}  PASS")

del model_test
print("\n  Architecture verified for all spectrogram sizes.")

---

## 5. Discover Machine Types

Each machine type gets its **own autoencoder** — we train 7 separate models.

In [ ]:
machine_types = sorted([
    d for d in os.listdir(PROCESSED_DIR)
    if os.path.isdir(os.path.join(PROCESSED_DIR, d))
])

print(f"Found {len(machine_types)} machine types:")
for i, m in enumerate(machine_types, 1):
    train_path = os.path.join(PROCESSED_DIR, m, "X_train.npy")
    if os.path.exists(train_path):
        shape = np.load(train_path, mmap_mode='r').shape
        print(f"  {i}. {m:<12s} - X_train shape: {shape}")
    else:
        print(f"  {i}. {m:<12s} - Missing X_train.npy")

---

## 6. The Training Function

### What Happens Each Epoch

1. **Training phase** (`model.train()`): Forward → MSE loss → Backprop → Update weights
2. **Validation phase** (`model.eval()`): Forward only → compute val loss (no gradients)
3. **Early Stopping**: Save best weights. Stop after `PATIENCE` epochs without improvement.
4. **LR Scheduling**: `ReduceLROnPlateau` halves LR when val loss plateaus.

### Loss Function: MSE (Mean Squared Error)

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (x_i - \hat{x}_i)^2$$

The DCASE 2024 baseline standard. Penalizes large errors quadratically, making anomalous
sounds produce dramatically higher scores than normal sounds.

In [ ]:
def train_one_machine(machine_name, X_train, X_val, config):
    n_samples, n_mels, n_frames = X_train.shape
    input_dim = n_mels * n_frames

    model = AcousticAutoencoder(input_dim, config['bottleneck_dim']).to(config['device'])
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config['lr'], weight_decay=config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=7, verbose=False
    )

    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train)),
        batch_size=config['batch_size'], shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_val)),
        batch_size=config['batch_size'], shuffle=False
    )

    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(config['num_epochs']):
        # --- TRAIN ---
        model.train()
        train_losses = []
        for (batch,) in train_loader:
            batch = batch.to(config['device'])
            output = model(batch)
            loss = criterion(output, batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # --- VALIDATE ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(config['device'])
                output = model(batch)
                loss = criterion(output, batch)
                val_losses.append(loss.item())

        avg_train = np.mean(train_losses)
        avg_val = np.mean(val_losses)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)

        scheduler.step(avg_val)
        current_lr = optimizer.param_groups[0]['lr']

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            marker = 'Best'
        else:
            epochs_no_improve += 1
            marker = f'No improve {epochs_no_improve}/{config["patience"]}'

        if (epoch + 1) % 5 == 0 or epochs_no_improve == 0 or epochs_no_improve >= config['patience']:
            print(f"    Epoch {epoch+1:3d}/{config['num_epochs']} | "
                  f"Train: {avg_train:.6f} | Val: {avg_val:.6f} | "
                  f"LR: {current_lr:.1e} | {marker}")

        if epochs_no_improve >= config['patience']:
            print(f"    Early stopping at epoch {epoch + 1}")
            break

    model.load_state_dict(best_model_state)
    return model, history


print("Training function defined.")

---

## 7. Train All Machine Types

Each model is saved to `models/<machine_type>/`.

In [ ]:
config = {
    'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'num_epochs': NUM_EPOCHS, 'patience': PATIENCE,
    'bottleneck_dim': BOTTLENECK_DIM, 'device': device,
}

all_histories = {}
training_summary = []
total_start = time.time()

for machine in machine_types:
    print(f"\n{'=' * 70}")
    print(f"TRAINING: {machine}")
    print(f"{'=' * 70}")

    X_train = np.load(os.path.join(PROCESSED_DIR, machine, "X_train.npy"))
    X_val   = np.load(os.path.join(PROCESSED_DIR, machine, "X_val.npy"))
    print(f"  Data: X_train={X_train.shape}  X_val={X_val.shape}")
    print(f"  Input dim: {X_train.shape[1]} x {X_train.shape[2]} = {X_train.shape[1] * X_train.shape[2]:,}")

    t0 = time.time()
    model, history = train_one_machine(machine, X_train, X_val, config)
    elapsed = time.time() - t0

    best_val = min(history['val_loss'])
    best_epoch = history['val_loss'].index(best_val) + 1

    save_dir = os.path.join(MODELS_DIR, machine)
    os.makedirs(save_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pth"))
    torch.save({
        'input_dim': model.input_dim, 'bottleneck_dim': BOTTLENECK_DIM,
        'n_mels': X_train.shape[1], 'n_frames': X_train.shape[2],
        'best_val_loss': best_val, 'best_epoch': best_epoch,
        'total_epochs': len(history['val_loss']),
    }, os.path.join(save_dir, "metadata.pth"))

    all_histories[machine] = history
    training_summary.append({
        'machine': machine, 'best_val_loss': best_val,
        'best_epoch': best_epoch, 'total_epochs': len(history['val_loss']),
        'time_sec': elapsed,
    })

    print(f"  Saved to {save_dir}")
    print(f"  Best val loss: {best_val:.6f} (epoch {best_epoch}) | Time: {elapsed:.1f}s")

total_elapsed = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"ALL TRAINING COMPLETE - Total time: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
print(f"{'=' * 70}")

---

## 8. Training Summary

In [ ]:
print(f"{'Machine':<12s} | {'Best Val Loss':>14s} | {'Best Epoch':>10s} | {'Total Epochs':>12s} | {'Time (s)':>8s}")
print("-" * 72)
for s in training_summary:
    print(f"{s['machine']:<12s} | {s['best_val_loss']:14.6f} | {s['best_epoch']:10d} | {s['total_epochs']:12d} | {s['time_sec']:8.1f}")

---

## 9. Training & Validation Loss Curves

- **Both curves decreasing** = healthy learning
- **Train decreasing, Val increasing** = overfitting
- **Red dashed line** = epoch where best validation loss occurred

In [ ]:
n_machines = len(all_histories)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for idx, (machine, history) in enumerate(all_histories.items()):
    if idx >= len(axes):
        break
    ax = axes[idx]
    epochs = range(1, len(history['train_loss']) + 1)
    ax.plot(epochs, history['train_loss'], label='Train', linewidth=1.5)
    ax.plot(epochs, history['val_loss'], label='Validation', linewidth=1.5)
    best_ep = history['val_loss'].index(min(history['val_loss'])) + 1
    ax.axvline(x=best_ep, color='red', linestyle='--', alpha=0.5, label=f'Best (ep {best_ep})')
    ax.set_title(machine, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for idx in range(n_machines, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Training & Validation Loss Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs(MODELS_DIR, exist_ok=True)
plt.savefig(os.path.join(MODELS_DIR, "training_curves.png"), dpi=150, bbox_inches='tight')
print("Saved training curves.")
plt.show()

---

## Summary

### What We Built

- Optimized Dense Autoencoder with BatchNorm, multi-layer Dropout, dynamic input dimensions
- Trained 7 separate models (one per machine type) with Early Stopping and LR scheduling
- Saved model weights and metadata for evaluation in Phase 3

### Next: Phase 3 — Evaluation (`03_evaluation.ipynb`)

Load each trained model, process test audio (normal + anomalous), compute anomaly scores,
and calculate AUC-ROC to measure detection performance.